# Lab 3: Sentinel-1 Time Series for Crop Growth in Midwest Row-Crop Fields

This notebook is for a lab on seasonal crop growth using Sentinel-1 SAR backscatter in large Midwestern U.S. fields.

The lab is designed to stay fully in Google Earth Engine + Python. We will be:

- building a consistent Sentinel-1 time series,
- selecting individual row-crop fields, forested regions, and towns/cities,
- extracting field-scale backscatter statistics,
- comparing seasonal trajectories,
- proposing a simple crop growth metric based on VV, VH, or VV/VH.

---

## Lab concept

Students will:

1. choose several individual regions in a Midwest US AOI. You do not have to choose a midwest US AOI, but it will be the easiest. Midwest US/Canada has perfectly North/South Aligned farms in 1.5 km grids, which uniformly grow corn/wheat/soy/etc. European farms are much smaller, and randomly oriented. Averaging over an entire field will increase the signal to noise ratio.
2. define field polygons as regions of interest,
3. build a seasonal Sentinel-1 time series,
4. compute field-level summary statistics for:
   - VV
   - VH
   - VV - VH (in dB is a convenient proxy related to VV/VH ratio)
5. interpret how those quantities evolve from:
   - bare ground / early season
   - vegetative growth
   - maturity
   - senescence (interesting fact - senescence progression is monitored via NDVI)
   - harvest

---

## Why the Midwest U.S.?

This is a good teaching region because many fields are:

- large,
- homogeneous,
- easy to outline,
- minimally affected by complex topography,
- dominated by row crops with strong seasonal structure.

That makes field-scale SAR time series much easier to interpret than small, irregular fields. Irregularly oriented fields produce an additional complexity with regards to geometry. You may have noticed when exploring Krakow SAR that the backscatter from farms could vary greatly. This can be due to differences in crop type, but it can also be related to how the rows are oriented with respect to the satellite look direction. Think about the ground texture of farms, there are many small ridges and valleys. If these ridges and valleys are parallel to the satellite look direction, backscatter will be weaker, and if the ridges and valleys are perpendicular to the satellite look direction, backscatter will be stronger.

---

## Learning goals

By the end of this lab, students should be able to:

- explain why SAR time series must be filtered consistently,
- compute a field-averaged VV and VH time series,
- compare crop fields with a non-crop control,
- propose and defend simple crop growth metrics,
- explain why VH or VV-VH may respond differently from VV alone.

---

## 1. Imports and Earth Engine initialization

This notebook assumes that students already have:

- a working Python environment,
- the Earth Engine Python API,
- `geemap`,
- libraries `matplotlib` and `pandas`.

In [1]:
import ee
import geemap
import pandas as pd
import json
import matplotlib.pyplot as plt

PROJECT_ID = "river-dynamo-494108-u6"

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized.")

Earth Engine initialized.


## 2. Define an agricultural AOI

This notebook starts with a large AOI in northern Iowa / southern Minnesota.  
You can adjust it later, but it is helpful to begin with a region dominated by large row-crop fields. In one scene, I want you to find a place that has:

- individual crop fields that you can isolate
- A city or town that we're not expecting a strong vegetative growth signal from
- a stand of trees you can isolate

**QUESTION (5pts): Why is the subregion you chose a good choice for exploring crop growth? Trees and crops both have vegetative growth during summer months, but what differences would you expect to see over a year, with regards to VV, VH, and VV-VH? What do you expect to see in the city or town that you have isolated?**

**ANSWER:**


In [21]:
# Example AOI: large row-crop region in northern Iowa / southern Minnesota
aoi = ee.Geometry.Rectangle([-93.95, 43.65, -93.25, 44.15])

aoi_trees = ee.Geometry.Rectangle([-93.81393, 43.86132, -93.80792, 43.85824])
aoi_field_1 = ee.Geometry.Rectangle([-93.81493, 43.84368, -93.80792, 43.83327])
aoi_field_2 = ee.Geometry.Rectangle([-93.79770, 43.87328, -93.78823, 43.86989])
aoi_field_3 = ee.Geometry.Rectangle([-93.76444, 43.88411, -93.76129, 43.87701])
aoi_city = ee.Geometry.Rectangle([-93.50694, 44.08000, -93.49697, 44.06647])

# Growing season year
YEAR = 2025

# Visualization parameters
vv_vis = {"min": -18, "max": 2}
vh_vis = {"min": -28, "max": -8}
ratio_vis = {"min": 2, "max": 12}

m = geemap.Map(center=[43.9, -93.6], zoom=10)
m.addLayer(aoi, {"color": "red"}, "AOI")
m.addLayer(aoi_trees, {"color": "green"}, "AOI_Trees")
m.addLayer(aoi_field_1, {"color": "yellow"}, "AOI_Field_1")
m.addLayer(aoi_field_2, {"color": "orange"}, "AOI_Field_2")
m.addLayer(aoi_field_3, {"color": "purple"}, "AOI_Field_3")
m.addLayer(aoi_city, {"color": "gray"}, "AOI_City")
m

Map(center=[43.9, -93.6], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

## 3. Helper functions

To keep the notebook readable, the repetitive Earth Engine logic is collected into helper functions.

These functions do most of the work for:

- building a filtered Sentinel-1 collection,
- creating monthly composites,
- computing a convenient VV-VH metric,
- extracting region statistics through time.

---

### Why use `VV - VH` here?

Sentinel-1 GRD backscatter in Earth Engine is in dB.  
So for this lab, instead of using a raw linear ratio `VV/VH`, we use:

$$
VV - VH
$$

because in dB space this is directly related to the log-ratio and is easier to interpret consistently alongside VV and VH.


In [17]:
def get_s1_collection(start_date, end_date, region=aoi,
                      orbit_pass="ASCENDING",
                      mode="IW",
                      resolution=10):
    """Return a filtered Sentinel-1 GRD collection with VV and VH."""
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(region)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("instrumentMode", mode))
        .filter(ee.Filter.eq("orbitProperties_pass", orbit_pass))
        .filter(ee.Filter.eq("resolution_meters", resolution))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
    )
    return collection


def monthly_composite(year, month, region=aoi):
    """Return a monthly median Sentinel-1 image with a derived VV-VH band."""
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")

    collection = get_s1_collection(start, end, region=region)
    image = collection.median().clip(region)

    vv_minus_vh = image.select("VV").subtract(image.select("VH")).rename("VV_minus_VH")

    return (
        image.addBands(vv_minus_vh)
        .set({
            "year": year,
            "month": month,
            "system:time_start": start.millis(),
            "scene_count": collection.size()
        })
    )


def build_monthly_stack(year, months=range(3, 11), region=aoi):
    """Create a monthly ImageCollection for a growing season."""
    images = [monthly_composite(year, month, region=region) for month in months]
    return ee.ImageCollection(images)


def region_time_series(image_collection, region, scale=10):
    """Extract field-level mean VV, VH, and VV_minus_VH through time.

    Accepts either an ee.Geometry or an ee.Feature.
    """
    # If region is a Feature, convert it to its geometry
    if isinstance(region, ee.feature.Feature):
        region = region.geometry()

    def per_image(img):
        stats = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=scale,
            maxPixels=1e9,
            bestEffort=True
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"),
            "year": img.get("year"),
            "month": img.get("month"),
            "scene_count": img.get("scene_count"),
            "VV": stats.get("VV"),
            "VH": stats.get("VH"),
            "VV_minus_VH": stats.get("VV_minus_VH")
        })

    return ee.FeatureCollection(image_collection.map(per_image))

def fc_to_df(fc):
    """Convert an Earth Engine FeatureCollection of stats into a pandas DataFrame."""
    data = fc.getInfo()["features"]
    rows = [f["properties"] for f in data]
    df = pd.DataFrame(rows)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
    return df


## 4. Build a seasonal Sentinel-1 time series

We build monthly median images from February through November.

**QUESTION (5pts): What are the advantages and disadvantages of using monthly median stacks instead of relying on single scenes? In what circumstances would you not want to do any temporal averaging, and instead rely on single scenes?**

**ANSWER:**

**QUESTION (5pts): Why do we filter Sentinel-1 by orbit direction, instrument mode, polarization, and resolution before building the time series? What might happen if ascending and descending passes were mixed in the same time series?**

**ANSWER:**


In [22]:
months = list(range(2, 12))  # February through November
monthly_stack = build_monthly_stack(YEAR, months=months, region=aoi)
monthly_stack_trees = build_monthly_stack(YEAR, months=months, region=aoi_trees)
monthly_stack_field_1 = build_monthly_stack(YEAR, months=months, region=aoi_field_1)
monthly_stack_field_2 = build_monthly_stack(YEAR, months=months, region=aoi_field_2)
monthly_stack_field_3 = build_monthly_stack(YEAR, months=months, region=aoi_field_3)
monthly_stack_city = build_monthly_stack(YEAR, months=months, region=aoi_city)

print("Monthly images in stack for trees:", monthly_stack_trees.size().getInfo())
print("Monthly images in stack for field:", monthly_stack_field_1.size().getInfo())
print("Monthly images in stack for city:", monthly_stack_city.size().getInfo())

# Check how many scenes contribute to each month
for month in months:
    img_trees = monthly_composite(YEAR, month, aoi_trees)
    img_field_1 = monthly_composite(YEAR, month, aoi_field_1)
    img_field_2 = monthly_composite(YEAR, month, aoi_field_2)
    img_field_3 = monthly_composite(YEAR, month, aoi_field_3)
    img_city = monthly_composite(YEAR, month, aoi_city)
    print(f"[TREES] Month {month:02d}: {ee.Number(img_trees.get('scene_count')).getInfo()} scenes")
    print(f"[FIELD] Month {month:02d}: {ee.Number(img_field_1.get('scene_count')).getInfo()} scenes")
    print(f"[CITY] Month {month:02d}: {ee.Number(img_city.get('scene_count')).getInfo()} scenes")


Monthly images in stack for trees: 10
Monthly images in stack for field: 10
Monthly images in stack for city: 10
[TREES] Month 02: 5 scenes
[FIELD] Month 02: 5 scenes
[CITY] Month 02: 5 scenes
[TREES] Month 03: 6 scenes
[FIELD] Month 03: 6 scenes
[CITY] Month 03: 6 scenes
[TREES] Month 04: 7 scenes
[FIELD] Month 04: 7 scenes
[CITY] Month 04: 7 scenes
[TREES] Month 05: 4 scenes
[FIELD] Month 05: 4 scenes
[CITY] Month 05: 4 scenes
[TREES] Month 06: 3 scenes
[FIELD] Month 06: 3 scenes
[CITY] Month 06: 3 scenes
[TREES] Month 07: 3 scenes
[FIELD] Month 07: 3 scenes
[CITY] Month 07: 3 scenes
[TREES] Month 08: 4 scenes
[FIELD] Month 08: 4 scenes
[CITY] Month 08: 4 scenes
[TREES] Month 09: 2 scenes
[FIELD] Month 09: 2 scenes
[CITY] Month 09: 2 scenes
[TREES] Month 10: 4 scenes
[FIELD] Month 10: 4 scenes
[CITY] Month 10: 4 scenes
[TREES] Month 11: 4 scenes
[FIELD] Month 11: 4 scenes
[CITY] Month 11: 4 scenes


## 5. Quick visual inspection

Before drawing fields, it is useful to inspect one or two monthly composites.

Suggested things to look for:

- field texture,
- strong contrasts between different fields,
- stable non-crop controls such as roads, towns, or tree patches.

**QUESTION (10pts): Compare an early month with a peak month visually. Feel free to change the months plotted, or add in comparisons of early vs mid- season VV_minus_VH, for your investigations. Take particular note of how much VH changes vs how much VV changes. What regions do you think you will choose for your ROI's and why? If you want an extra challenge (but not necessary), try to find a field you think was tall crops, a field you think was short crops, and a field you think was left barren.**

**ANSWER:**

In [29]:
img_early = monthly_composite(YEAR, 2, aoi)
img_peak = monthly_composite(YEAR, 7, aoi)

img_field_1_early = monthly_composite(YEAR, 2, aoi_field_1)
img_field_1_peak = monthly_composite(YEAR, 7, aoi_field_1)

img_field_2_early = monthly_composite(YEAR, 2, aoi_field_2)
img_field_2_peak = monthly_composite(YEAR, 7, aoi_field_2)

img_field_3_early = monthly_composite(YEAR, 2, aoi_field_3)
img_field_3_peak = monthly_composite(YEAR, 7, aoi_field_3)

img_trees_early = monthly_composite(YEAR, 2, aoi_trees)
img_trees_peak = monthly_composite(YEAR, 7, aoi_trees)

img_city_early = monthly_composite(YEAR, 2, aoi_city)
img_city_peak = monthly_composite(YEAR, 7, aoi_city)

#YOUR CODE HERE: The lines above grab the montlhy composite images for February and July. I want you to plot a split_map image of the VH channel in this cell, and in the cell below plot a split_map image of the VV channel for the same months.
m = geemap.Map(center=[43.9, -93.6], zoom=11)
m.split_map(
    left_layer=geemap.ee_tile_layer(img_field_1_early.select("VH"), vh_vis, "VH February"),
    right_layer=geemap.ee_tile_layer(img_field_1_peak.select("VH"), vh_vis, "VH July")
)
display(m)

m_field_2 = geemap.Map(center=[43.9, -93.6], zoom=11)
m_field_2.split_map(
    left_layer=geemap.ee_tile_layer(img_field_2_early.select("VH"), vh_vis, "VH February"),
    right_layer=geemap.ee_tile_layer(img_field_2_peak.select("VH"), vh_vis, "VH July")
)
display(m_field_2)

m_field_3 = geemap.Map(center=[43.9, -93.6], zoom=11)
m_field_3.split_map(
    left_layer=geemap.ee_tile_layer(img_field_3_early.select("VH"), vh_vis, "VH February"),
    right_layer=geemap.ee_tile_layer(img_field_3_peak.select("VH"), vh_vis, "VH July")
)
display(m_field_3)

m_city = geemap.Map(center=[43.9, -93.6], zoom=11)
m_city.split_map(
    left_layer=geemap.ee_tile_layer(img_city_early.select("VH"), vh_vis, "VH February"),
    right_layer=geemap.ee_tile_layer(img_city_peak.select("VH"), vh_vis, "VH July")
)
display(m_city)

m_trees = geemap.Map(center=[43.9, -93.6], zoom=11)
m_trees.split_map(
    left_layer=geemap.ee_tile_layer(img_trees_early.select("VH"), vh_vis, "VH February"),
    right_layer=geemap.ee_tile_layer(img_trees_peak.select("VH"), vh_vis, "VH July")
)
display(m_trees)
#m.addLayer(aoi, {"color": "red"}, "AOI")

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [30]:
#Plot the VV split_map here.

m = geemap.Map(center=[43.9, -93.6], zoom=11)
m.split_map(
    left_layer=geemap.ee_tile_layer(img_field_1_early.select("VV"), vh_vis, "VV February"),
    right_layer=geemap.ee_tile_layer(img_field_1_peak.select("VV"), vh_vis, "VV July")
)
display(m)

m_field_2 = geemap.Map(center=[43.9, -93.6], zoom=11)
m_field_2.split_map(
    left_layer=geemap.ee_tile_layer(img_field_2_early.select("VV"), vh_vis, "VV February"),
    right_layer=geemap.ee_tile_layer(img_field_2_peak.select("VV"), vh_vis, "VV July")
)
display(m_field_2)

m_field_3 = geemap.Map(center=[43.9, -93.6], zoom=11)
m_field_3.split_map(
    left_layer=geemap.ee_tile_layer(img_field_3_early.select("VV"), vh_vis, "VV February"),
    right_layer=geemap.ee_tile_layer(img_field_3_peak.select("VV"), vh_vis, "VV July")
)
display(m_field_3)

m_city = geemap.Map(center=[43.9, -93.6], zoom=11)
m_city.split_map(
    left_layer=geemap.ee_tile_layer(img_city_early.select("VV"), vh_vis, "VV February"),
    right_layer=geemap.ee_tile_layer(img_city_peak.select("VV"), vh_vis, "VV July")
)
display(m_city)

m_trees = geemap.Map(center=[43.9, -93.6], zoom=11)
m_trees.split_map(
    left_layer=geemap.ee_tile_layer(img_trees_early.select("VV"), vh_vis, "VV February"),
    right_layer=geemap.ee_tile_layer(img_trees_peak.select("VV"), vh_vis, "VV July")
)
display(m_trees)

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Map(center=[43.9, -93.6], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

# 6. Draw field polygons

Use the map drawing tools to create several polygons:

- at least **two different crop fields**
- at least **two control areas** (1 patch of trees, 1 city or town)

### Suggested strategy
Pick fields that are:

- large,
- internally uniform,
- clearly separated from neighboring fields.

After drawing polygons on the map, export them to Earth Engine objects.

**NOTE: You will need to remember the order of the ROI's you chose. For example, remember that you first chose three farm fields, then a tree stand, then the city/town. In the next section, you will need to assign these drawn features to variables and a dictionary.**



In [7]:
m = geemap.Map(center=[43.9, -93.6], zoom=12)
m.addLayer(img_peak.select("VV"), vv_vis, "VV July")
m.addLayer(img_peak.select("VH"), vh_vis, "VH July")
#m.addLayer(aoi, {"color": "red"}, "AOI")
m

# After drawing polygons with the map tools, use:
# drawn_fc = m.draw_features

Map(center=[43.9, -93.6], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [9]:
drawn_fc = m.draw_features
print(drawn_fc)

[]


## 7. Make sure your drawn features are where you want them.

To do this, you will need to need to adjust the code below to match the order and number of ROI's you chose to investigate.


In [10]:
# Example fallback polygons
#field_1 = ee.Geometry.Rectangle([-93.731, 43.949, -93.714, 43.964])
#field_2 = ee.Geometry.Rectangle([-93.666, 43.865, -93.646, 43.882])
#control_1 = ee.Geometry.Rectangle([-93.585, 43.971, -93.571, 43.982])  # mixed non-crop / built area
field_1 = drawn_fc[0]
field_2 = drawn_fc[1]
field_3 = drawn_fc[2]
trees_1 = drawn_fc[3]
control_1 = drawn_fc[4]
roi_dict = {
    "Field_1": field_1,
    "Field_2": field_2,
    "Field_3": field_3,
    "Trees_1": trees_1,
    "Control_1": control_1
}

m = geemap.Map(center=[43.9, -93.6], zoom=12)
m.addLayer(img_peak.select("VV"), vv_vis, "VV July")
for name, geom in roi_dict.items():
    m.addLayer(geom, {"color": "yellow"}, name)
m

##PLEASE PASTE THE OUTPUT OF THIS AS A COMMENT HERE SO THAT I KNOW WHICH ROI's YOU CHOSE
for name, roi in roi_dict.items():
    print(name, roi.geometry().getInfo())


IndexError: list index out of range

## 8. Extract time series for each field

For each ROI, compute a field-mean time series of:

- VV
- VH
- VV - VH

This is the core analysis step of the lab.

In [ ]:
ts_dict = {}

for name, geom in roi_dict.items():
    fc = region_time_series(monthly_stack, geom, scale=10)
    df = fc_to_df(fc)
    ts_dict[name] = df

ts_dict["Field_1"]


**QUESTION (5pts): Which month has the minimum VH? Which month has the maximum VH? Does this match what you expected from the image comparison?**

**ANSWER:**


## 9. Plot VV, VH, and VV-VH through time

These plots are where the crop-growth story should begin to emerge.

**QUESTION (20pts):**
- Qualitatively compare your three different types of ROI's (1) farms; (2) trees; and (3) towns. A few sentences to a short paragraph is enough.
- Based on these plots, identify the months that (1) crops started growing; (2) crops stopped growing; and (3) crops were harvested.
- Does VV-VH separate crops from control AOI's better than VV alone?
- Which ROI looks least like an actively growing crop field?
- Which ROI shows the strongest seasonal change in VH? Which ROI is most stable through time? Explain whether each behavior is more consistent with crop growth, forest canopy, or built-up land.

**ANSWER:**


In [1]:
fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

for name, df in ts_dict.items():
    axes[0].plot(df["date"], df["VV"], marker="o", label=name)
    axes[1].plot(df["date"], df["VH"], marker="o", label=name)
    axes[2].plot(df["date"], df["VV_minus_VH"], marker="o", label=name)

axes[0].set_title("Field-mean VV through the growing season")
axes[0].set_ylabel("VV (dB)")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Field-mean VH through the growing season")
axes[1].set_ylabel("VH (dB)")
axes[1].grid(True, alpha=0.3)

axes[2].set_title("Field-mean VV - VH through the growing season")
axes[2].set_ylabel("VV - VH (dB)")
axes[2].set_xlabel("Date")
axes[2].grid(True, alpha=0.3)

axes[0].legend()
plt.tight_layout()
plt.show()


NameError: name 'plt' is not defined

## 10. Build a simple crop-growth metric

Next I want you to propose a metric that captures crop development.

Possible first ideas include:

- peak VH minus spring VH,
- July VH minus April VH,
- maximum seasonal VV-VH range,
- cumulative increase from planting to peak growth.

Below is one simple example:

\[
Growth Metric = VH_{peak} - VH_{early}
\]

This is not necessarily the best metric — it is just a starting point.



In [1]:
##YOUR CODE HERE (50pts)


## 11. Optional: compare fields spatially for one month

It is often useful to look at the mapped field values in the month of strongest contrast.

This can help connect your field-average statistics back to the actual imagery.


In [ ]:
img_july = monthly_composite(YEAR, 7, aoi)

m = geemap.Map(center=[43.9, -93.6], zoom=12)
m.addLayer(img_july.select("VV"), vv_vis, "VV July")
m.addLayer(img_july.select("VH"), vh_vis, "VH July")
m.addLayer(img_july.select("VV_minus_VH"), ratio_vis, "VV-VH July")

for name, geom in roi_dict.items():
    m.addLayer(geom, {"color": "yellow"}, name)

m

## 12. Optional: identify likely tall vs shorter crops

With Sentinel-1 alone, you should not make an overconfident crop-type classification.  
But you can propose that one field is more consistent with a tall, strongly volume-scattering canopy than another.

For example:
- stronger seasonal VH increase,
- larger mid-season drop in VV-VH,
- more pronounced growth curve.


## Final note
This lab is designed to connect a simple SAR observable to a real seasonal process.  
The goal is not perfect crop classification, but rather to help reason from:

- backscatter,
- polarization,
- time,
- and field geometry

toward a physically grounded interpretation of crop growth.
